# 03 Feature Extraction

In [ ]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
import sys
sys.path.append('/content/unet-ensemble')

## 03-1 Frequency Domain

## 03-2 Illumination Patterns

Illumination Pattern Extraction with Local Statistical Analysis

**In extraction, the following are used:**
1. Grayscale Conversion (cv2.COLOR_BGR2GRAY, float64 precision)
2. Local Mean Estimation (Gaussian filtering using gaussian_filter, sigma = 80)
3. Local Variance Estimation (Box filtering using uniform_filter, window size = 31)

```
# illumination pattern extraction
local_mean = gaussian_filter(image, sigma=sigma)

lm_box = uniform_filter(image, size=window_size)
lm_sq  = uniform_filter(image ** 2, size=window_size)

local_var = lm_sq - lm_box ** 2
illum = normalize(local_mean * local_var)
```
This assumes that illumination characteristics can be represented through local intensity statistics, where:

$M(x,y)$ represents the local mean illumination, and
$V(x,y)$ represents the local variance of illumination changes.

So the illumination pattern is approximated as:

$L(x,y) = M(x,y) \times V(x,y)$

**After extraction, these are applied:**
1. Min-max normalization (scale intermediate maps to 0–255 uint8)

**For visualization:**
1. Direct display via plt.imshow(cmap='gray') — no additional enhancement applied

In [ ]:
import os
import cv2
from src.features.illumination import Illumination

In [ ]:
SIZE        = 256
SIGMA       = 80
WINDOW_SIZE = 31

INPUT_DIR = '/content/drive/Shareddrives/U-Net Ensemble/Dataset/Evaluation/Normalized/Authentic/template-a'
OUTPUT_DIR = '/content/drive/Shareddrives/U-Net Ensemble/Dataset/Evaluation/Feature - Illumination/Authentic/template-a'

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
illum = Illumination()

for filename in os.listdir(INPUT_DIR):
  if filename.lower().endswith((".jpg", ".png", ".jpeg")):
    img_path = os.path.join(INPUT_DIR, filename)

    img = illum.load_image(img_path)
    
    if img is None:
      print(f"Could not read image: {filename}, skipping.")
      continue

    mean = illum.get_mean(img, SIGMA)
    var = illum.get_variance(img, WINDOW_SIZE)
    combined = illum.blend(mean, var)

    cv2.imwrite(os.path.join(OUTPUT_DIR, filename), combined)
    print(f"Processed: {filename}")

## 03-3 Photo-Response Non-Uniformity

PRNU Extraction with SPN (Sensor Pattern Noise) Refinement

**In extraction, the following are used:**
1. Mild NL-Means Denoising (h=2, templateWindow=5, searchWindow=21)
2. Wavelet-Based Denoising (BayesShrink, db4, 4 levels)
3. Additive Noise Model (for PRNU Residual Computation)

```
# additive noise model
residual = gray - denoised
```

This assumes: $I = I_0 + K + \Theta$ <br>

So: $K ≈ I - F(I)$ <br>

**After extraction, these are applied:**
1. Gausian filtering (remove low-frequency leakage)
2. Column mean suppression (remove banding)
3. Clip to ±3 std (remove extreme outliers)

**For visualization:**
1. Min-max stretch (scale to 0-255 uint8)
2. CLAHE (clipLimit=2, tileGrid=8x8), for local contrast enhancement
3. Gamma correction (y=0.6), for brigthness adjustment

In [5]:
import os
import cv2
from src.features.prnu import PRNU

In [ ]:
INPUT_DIR = '/content/drive/Shareddrives/U-Net Ensemble/Dataset/Evaluation/Normalized/Authentic/template-a'
OUTPUT_DIR = '/content/drive/Shareddrives/U-Net Ensemble/Dataset/Evaluation/Feature - PRNU/Authentic/template-a'

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
for filename in os.listdir(INPUT_DIR):
  if filename.lower().endswith((".jpg", ".png", ".jpeg")):
    img_path = os.path.join(INPUT_DIR, filename)
    
    prnu = PRNU(img_path)
    img = prnu.load_image()

    if img is None:
      print(f"Could not read image: {filename}, skipping.")
      continue

    wavelet, means = prnu.denoise_image(img)
    residual = prnu.suppress_residual(wavelet, means)

    vis = prnu.visualize(residual)

    cv2.imwrite(os.path.join(OUTPUT_DIR, filename), vis)
    print(f"Processed: {filename}")